# Debug Notebook for `05_label_llm.py`

This notebook tests the production LLM-labeling logic before running the full script.

Updated design:

1. Read unlabeled rows from `reddit_sentence_items`
2. Build a context window from:
   - preceding sentence
   - AI sentence
   - subsequent sentence
3. Combine the stance prompt and metaphor prompt into one unified labeling prompt
4. Use `llama3.2` through Ollama
5. Make one LLM call per row
6. Parse one JSON output containing:
   - `granularity`
   - `stance`
   - `dominant_metaphor`
7. Insert clean outputs into `llm_labels`
8. Confirm the pipeline is safe to rerun without duplicates


## 1. Imports and configuration

In [5]:
from __future__ import annotations

import json
import logging
import re
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import pandas as pd
import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

SOURCE = "reddit"

DB_PATH = Path("../data/database/framescope.db")

METAPHOR_PROMPT_PATH = Path("../prompts/metaphor_classification_v1.txt")
STANCE_PROMPT_PATH = Path("../prompts/stance_classification_v1.txt")

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2"

BATCH_SIZE = 20
MAX_WORKERS = 2
REQUEST_TIMEOUT = 120

LLM_CONFIG = {
    "temperature": 0.0,
    "top_p": 1.0,
    "top_k": 20,
    "repeat_penalty": 1.0,
    "num_predict": 200,
}

VALID_METAPHORS = {
    "Tool",
    "Assistant",
    "Genie",
    "Mirror",
    "Child",
    "Friend",
    "Animal",
    "God",
    "None",
}

VALID_GRANULARITY = {
    "General-AI",
    "Model-Specific",
    "Domain-Specific",
    "Not Applicable",
}

VALID_STANCE = {
    "Positive",
    "Neutral/Unclear",
    "Negative",
}


## 2. Database and prompt helpers

In [6]:
def connect_db(db_path: Path) -> sqlite3.Connection:
    if not db_path.exists():
        raise FileNotFoundError(f"Database not found: {db_path}")

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA foreign_keys=ON;")
    return conn


def ensure_label_schema(conn: sqlite3.Connection) -> None:
    conn.executescript(
        '''
        CREATE TABLE IF NOT EXISTS llm_labels (
            source TEXT NOT NULL,
            sentence_id TEXT NOT NULL,
            metaphor_category TEXT,
            metaphor_present INTEGER,
            granularity TEXT,
            stance TEXT,
            confidence REAL,
            reasoning TEXT,
            model_name TEXT,
            labeled_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (source, sentence_id)
        );
        '''
    )

    existing_cols = {
        row[1] for row in conn.execute("PRAGMA table_info(llm_labels);").fetchall()
    }

    if "granularity" not in existing_cols:
        conn.execute("ALTER TABLE llm_labels ADD COLUMN granularity TEXT;")

    conn.commit()


def load_prompt(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing prompt file: {path}")

    return path.read_text(encoding="utf-8")


## 3. Connect to database and load prompt files

In [7]:
conn = connect_db(DB_PATH)
ensure_label_schema(conn)

metaphor_prompt_template = load_prompt(METAPHOR_PROMPT_PATH)
stance_prompt_template = load_prompt(STANCE_PROMPT_PATH)

print("Connected to:", DB_PATH)
print("Loaded metaphor prompt:", METAPHOR_PROMPT_PATH)
print("Loaded stance prompt:", STANCE_PROMPT_PATH)


Connected to: ../data/database/framescope.db
Loaded metaphor prompt: ../prompts/metaphor_classification_v1.txt
Loaded stance prompt: ../prompts/stance_classification_v1.txt


## 4. Check Ollama

In [8]:
def check_ollama() -> None:
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=10)
        response.raise_for_status()
        print("Ollama is running.")
        print(response.json())
    except Exception as exc:
        raise RuntimeError(
            "Ollama is not running. Start it with: ollama serve"
        ) from exc

check_ollama()


Ollama is running.
{'models': [{'name': 'qwen2:latest', 'model': 'qwen2:latest', 'modified_at': '2026-04-24T08:29:42.629462579-04:00', 'size': 4431401491, 'digest': 'dd314f039b9d54d5553002c906ce50c9fe7242f73f0680abd04f01c8ecbd2755', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '7.6B', 'quantization_level': 'Q4_0'}}, {'name': 'phi3:latest', 'model': 'phi3:latest', 'modified_at': '2026-04-24T08:27:48.920553779-04:00', 'size': 2176178913, 'digest': '4f222292793889a9a40a020799cfd28d53f3e01af25d48e06c5e708610fc47e9', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'phi3', 'families': ['phi3'], 'parameter_size': '3.8B', 'quantization_level': 'Q4_0'}}, {'name': 'gemma:7b', 'model': 'gemma:7b', 'modified_at': '2026-04-24T08:26:50.020128816-04:00', 'size': 5011853225, 'digest': 'a72c7f4d0a15522df81486d13ce432c79e191bda2558d024fbad4362c4f7cbf8', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma', 'famil

## 5. Fetch unlabeled rows

This assumes `reddit_sentence_items` contains columns named:

- `preceding_sentence`
- `ai_sentence`
- `subsequent_sentence`

If your table uses different names, adjust the SQL query below.


In [9]:
def fetch_unlabeled_rows(conn: sqlite3.Connection, limit: int) -> list[dict[str, Any]]:
    query = '''
        SELECT
            r.sentence_id,
            r.preceding_sentence,
            r.ai_sentence,
            r.subsequent_sentence,
            r.context_text,
            r.subreddit,
            r.created_utc,
            r.score
        FROM reddit_sentence_items r
        LEFT JOIN llm_labels l
            ON r.sentence_id = l.sentence_id
            AND l.source = ?
        WHERE l.sentence_id IS NULL
        ORDER BY r.created_utc ASC
        LIMIT ?;
    '''

    rows = conn.execute(query, (SOURCE, limit)).fetchall()

    return [
        {
            "sentence_id": row[0],
            "preceding_sentence": row[1],
            "ai_sentence": row[2],
            "subsequent_sentence": row[3],
            "context_text": row[4],
            "subreddit": row[5],
            "created_utc": row[6],
            "score": row[7],
        }
        for row in rows
    ]


rows = fetch_unlabeled_rows(conn, limit=5)
print("Fetched rows:", len(rows))
pd.DataFrame(rows).head()


Fetched rows: 5


,sentence_id,preceding_sentence,ai_sentence,subsequent_sentence,context_text,subreddit,created_utc,score
0,10074g2_s0000,None,AI Generated Content and Reddit - With the eme...,", what is Reddit position on using this conten...",AI Generated Content and Reddit - With the eme...,AskReddit,1672531224,1
1,10074g2_s0002,", what is Reddit position on using this conten...","For example, I used a book review generated by...",None,", what is Reddit position on using this conten...",AskReddit,1672531224,1
2,100753i_s0000,None,AI Generated Content and Reddit - With the eme...,", what is Reddit position on using this conten...",AI Generated Content and Reddit - With the eme...,AskReddit,1672531280,1
3,100753i_s0002,", what is Reddit position on using this conten...","For example, I used a book review generated by...",None,", what is Reddit position on using this conten...",AskReddit,1672531280,1
4,1007ad8_s0000,None,What are the potential consequences of artific...,None,What are the potential consequences of artific...,AskReddit,1672531735,1


## 6. Build context window and combined prompt

We combine stance and metaphor instructions, but force one final JSON schema to avoid conflicting output instructions.


In [10]:
def build_context_text(row: dict[str, Any]) -> str:
    parts = [
        row.get("preceding_sentence"),
        row.get("ai_sentence"),
        row.get("subsequent_sentence"),
    ]

    parts = [str(p).strip() for p in parts if p is not None and str(p).strip()]

    if parts:
        return " ".join(parts)

    fallback = row.get("context_text") or row.get("ai_sentence") or ""
    return str(fallback).strip()


def build_combined_prompt(
    text: str,
    metaphor_template: str,
    stance_template: str,
) -> str:
    # Remove separate final output ambiguity by adding one final unified schema.
    combined = f'''
You are labeling Reddit text related to artificial intelligence.

You will use the following stance/granularity coding guide:

{stance_template}

You will also use the following metaphor coding guide:

{metaphor_template}

IMPORTANT FINAL OUTPUT INSTRUCTION:
Return ONLY valid JSON in this exact format:

{{
  "granularity": "General-AI | Model-Specific | Domain-Specific | Not Applicable",
  "stance": "Positive | Neutral/Unclear | Negative",
  "dominant_metaphor": "Tool | Assistant | Genie | Mirror | Child | Friend | Animal | God | None"
}}

Rules:
- Do not include explanations.
- Do not include markdown.
- Use the exact label names shown above.
- If the text is not meaningfully about AI, use:
  - granularity: "Not Applicable"
  - stance: "Neutral/Unclear"
  - dominant_metaphor: "None"

TEXT:
{text}
'''

    return (
        combined
        .replace("{text}", text)
        .replace("{input_text}", text)
    )


sample_text = build_context_text(rows[0]) if rows else ""
print(sample_text[:1000])

sample_prompt = build_combined_prompt(
    sample_text,
    metaphor_prompt_template,
    stance_prompt_template,
)

print("\n--- Prompt preview ---\n")
print(sample_prompt[:3000])


AI Generated Content and Reddit - With the emergence of open AI sources such as ChatGPT , what is Reddit position on using this content for posts on Reddit.

--- Prompt preview ---


You are labeling Reddit text related to artificial intelligence.

You will use the following stance/granularity coding guide:

You are labeling Reddit text related to artificial intelligence.

Your task is to assign TWO labels:
1. granularity
2. stance

Return ONLY valid JSON in this exact format:

{
  "granularity": "one_label",
  "stance": "one_label"
}

----------------------------------------
GRANULARITY LABELS
----------------------------------------

General-AI:
Use this when the text discusses AI broadly, including general opinions, benefits, risks, or societal impacts of AI. The text is not focused on a specific model or a specific domain.

Model-Specific:
Use this when the text refers to a specific AI model, tool, or company system (e.g., ChatGPT, GPT-4, Claude, Gemini, LLaMA, Midjourney, Copilot,

## 7. Ollama call and output parsers

In [11]:
def call_ollama(prompt: str) -> str:
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": LLM_CONFIG,
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    return response.json().get("response", "").strip()


def extract_json(raw: str) -> dict[str, Any]:
    if not raw:
        return {}

    raw = raw.strip()

    try:
        return json.loads(raw)
    except Exception:
        pass

    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}

    return {}


def clean_metaphor(value: Any) -> str:
    if not isinstance(value, str):
        return "None"

    value = value.strip().replace('"', "").replace(".", "").replace(",", "")
    value = value.split("\n")[0].strip()

    lookup = {m.lower(): m for m in VALID_METAPHORS}

    return lookup.get(value.lower(), "None")


def clean_granularity(value: Any) -> str:
    if not isinstance(value, str):
        return "Not Applicable"

    value = value.strip()

    lookup = {
        "general-ai": "General-AI",
        "general_ai": "General-AI",
        "general ai": "General-AI",
        "general": "General-AI",
        "model-specific": "Model-Specific",
        "model_specific": "Model-Specific",
        "model specific": "Model-Specific",
        "domain-specific": "Domain-Specific",
        "domain_specific": "Domain-Specific",
        "domain specific": "Domain-Specific",
        "not applicable": "Not Applicable",
        "not_applicable": "Not Applicable",
        "na": "Not Applicable",
        "n/a": "Not Applicable",
    }

    if value in VALID_GRANULARITY:
        return value

    return lookup.get(value.lower(), "Not Applicable")


def clean_stance(value: Any, granularity: str) -> str:
    if granularity == "Not Applicable":
        return "Neutral/Unclear"

    if not isinstance(value, str):
        return "Neutral/Unclear"

    value = value.strip()

    lookup = {
        "positive": "Positive",
        "neutral": "Neutral/Unclear",
        "unclear": "Neutral/Unclear",
        "neutral/unclear": "Neutral/Unclear",
        "neutral_unclear": "Neutral/Unclear",
        "neutral unclear": "Neutral/Unclear",
        "negative": "Negative",
    }

    if value in VALID_STANCE:
        return value

    return lookup.get(value.lower(), "Neutral/Unclear")


def parse_combined_output(raw_output: str) -> tuple[str, str, str, dict[str, Any]]:
    parsed = extract_json(raw_output)

    metaphor = clean_metaphor(
        parsed.get("dominant_metaphor")
        or parsed.get("metaphor_category")
        or parsed.get("metaphor")
    )

    granularity = clean_granularity(parsed.get("granularity"))
    stance = clean_stance(parsed.get("stance"), granularity)

    return metaphor, granularity, stance, parsed


## 8. Single-row sanity test

Run this before doing any DB insert.


In [17]:
def label_one_row(
    row: dict[str, Any],
    metaphor_prompt_template: str,
    stance_prompt_template: str,
) -> dict[str, Any]:
    text = build_context_text(row)

    combined_prompt = build_combined_prompt(
        text=text,
        metaphor_template=metaphor_prompt_template,
        stance_template=stance_prompt_template,
    )

    try:
        start = time.time()

        raw_output = call_ollama(combined_prompt)

        latency = time.time() - start

        metaphor, granularity, stance, parsed = parse_combined_output(raw_output)

        return {
            "source": SOURCE,
            "sentence_id": row["sentence_id"],
            "context_text_used": text,
            "metaphor_category": metaphor,
            "metaphor_present": 0 if metaphor == "None" else 1,
            "granularity": granularity,
            "stance": stance,
            "confidence": None,
            "reasoning": None,
            "model_name": MODEL_NAME,
            "latency_seconds": latency,
            "error": None,
            "raw_output": raw_output,
            "parsed_output": parsed,
        }

    except Exception as exc:
        return {
            "source": SOURCE,
            "sentence_id": row["sentence_id"],
            "context_text_used": text,
            "metaphor_category": "None",
            "metaphor_present": 0,
            "granularity": "Not Applicable",
            "stance": "Neutral/Unclear",
            "confidence": None,
            "reasoning": None,
            "model_name": MODEL_NAME,
            "latency_seconds": None,
            "error": str(exc),
            "raw_output": None,
            "parsed_output": None,
        }


single_row = fetch_unlabeled_rows(conn, limit=1)[0]

single_result = label_one_row(
    single_row,
    metaphor_prompt_template,
    stance_prompt_template,
)

single_result


{'source': 'reddit',
 'sentence_id': '10074g2_s0000',
 'context_text_used': 'AI Generated Content and Reddit - With the emergence of open AI sources such as ChatGPT , what is Reddit position on using this content for posts on Reddit.',
 'metaphor_category': 'None',
 'metaphor_present': 0,
 'granularity': 'General-AI',
 'stance': 'Neutral/Unclear',
 'confidence': None,
 'reasoning': None,
 'model_name': 'llama3.2',
 'latency_seconds': 1.7623012065887451,
 'error': None,
 'raw_output': '{\n  "granularity": "General-AI",\n  "stance": "Neutral/Unclear",\n  "dominant_metaphor": "None"\n}',
 'parsed_output': {'granularity': 'General-AI',
  'stance': 'Neutral/Unclear',
  'dominant_metaphor': 'None'}}

## 9. Small batch test without database insert

This labels 10 rows but does not write them to the database.


In [20]:
test_rows = fetch_unlabeled_rows(conn, limit=10)

test_results = [
    label_one_row(r, metaphor_prompt_template, stance_prompt_template)
    for r in test_rows
]

test_df = pd.DataFrame(test_results)

test_df[
    [
        "sentence_id",
        "metaphor_category",
        "metaphor_present",
        "granularity",
        "stance",
        "latency_seconds",
        "error",
    ]
]


,sentence_id,metaphor_category,metaphor_present,granularity,stance,latency_seconds,error
0,10074g2_s0000,None,0,General-AI,Neutral/Unclear,0.888276,None
1,10074g2_s0002,None,0,General-AI,Neutral/Unclear,1.720587,None
2,100753i_s0000,None,0,General-AI,Neutral/Unclear,1.662268,None
3,100753i_s0002,None,0,General-AI,Neutral/Unclear,1.699893,None
4,1007ad8_s0000,Genie,1,General-AI,Positive,1.511137,None
5,1008450_s0000,None,0,General-AI,Neutral/Unclear,1.649208,None
6,10087h2_s0000,Genie,1,Model-Specific,Positive,1.447967,None
7,10090dy_s0000,None,0,Not Applicable,Neutral/Unclear,1.767781,None
8,100djuo_s0000,None,0,Not Applicable,Neutral/Unclear,1.527925,None
9,100dyxs_s0000,None,0,Not Applicable,Neutral/Unclear,0.743532,None


## 10. Inspect raw outputs

Use this to debug parsing issues.


In [22]:
for i, row in test_df.head(5).iterrows():
    print("=" * 100)
    print("sentence_id:", row["sentence_id"])
    print("context:", row["context_text_used"][:500])
    print("\nraw_output:")
    print(row["raw_output"])
    print("\nparsed_output:")
    print(row["parsed_output"])


sentence_id: 10074g2_s0000
context: AI Generated Content and Reddit - With the emergence of open AI sources such as ChatGPT , what is Reddit position on using this content for posts on Reddit.

raw_output:
{
  "granularity": "General-AI",
  "stance": "Neutral/Unclear",
  "dominant_metaphor": "None"
}

parsed_output:
{'granularity': 'General-AI', 'stance': 'Neutral/Unclear', 'dominant_metaphor': 'None'}
sentence_id: 10074g2_s0002
context: , what is Reddit position on using this content for posts on Reddit. For example, I used a book review generated by ChatGPT, which was a reasonably accurate review I felt, however the moderators determined it not

raw_output:
{
  "granularity": "General-AI",
  "stance": "Neutral/Unclear",
  "dominant_metaphor": "None"
}

parsed_output:
{'granularity': 'General-AI', 'stance': 'Neutral/Unclear', 'dominant_metaphor': 'None'}
sentence_id: 100753i_s0000
context: AI Generated Content and Reddit - With the emergence of open AI sources such as ChatGPT , what i

## 11. Validate label distributions from test batch

In [23]:
print("Metaphor distribution")
display(test_df["metaphor_category"].value_counts(dropna=False))

print("Granularity distribution")
display(test_df["granularity"].value_counts(dropna=False))

print("Stance distribution")
display(test_df["stance"].value_counts(dropna=False))

print("Errors")
display(test_df[test_df["error"].notna()])


Metaphor distribution


metaphor_category
None     8
Genie    2
Name: count, dtype: int64

Granularity distribution


granularity
General-AI        6
Not Applicable    3
Model-Specific    1
Name: count, dtype: int64

Stance distribution


stance
Neutral/Unclear    8
Positive           2
Name: count, dtype: int64

Errors


,source,sentence_id,context_text_used,metaphor_category,metaphor_present,granularity,stance,confidence,reasoning,model_name,latency_seconds,error,raw_output,parsed_output


## 12. Insert test batch into database

Only run this once you are satisfied with the small-batch outputs.


In [24]:
def insert_labels(conn: sqlite3.Connection, labeled_rows: list[dict[str, Any]]) -> int:
    before = conn.total_changes

    rows_to_insert = [
        (
            r["source"],
            r["sentence_id"],
            r["metaphor_category"],
            r["metaphor_present"],
            r["granularity"],
            r["stance"],
            r["confidence"],
            r["reasoning"],
            r["model_name"],
        )
        for r in labeled_rows
        if r.get("error") is None
    ]

    conn.executemany(
        '''
        INSERT OR IGNORE INTO llm_labels (
            source,
            sentence_id,
            metaphor_category,
            metaphor_present,
            granularity,
            stance,
            confidence,
            reasoning,
            model_name
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
        ''',
        rows_to_insert,
    )

    conn.commit()

    return conn.total_changes - before


# Uncomment when ready:
# inserted = insert_labels(conn, test_results)
# print("Inserted:", inserted)


## 13. Inspect database labels

In [25]:
query = '''
SELECT *
FROM llm_labels
WHERE source = ?
ORDER BY labeled_at DESC
LIMIT 10;
'''

labels_preview = pd.read_sql_query(query, conn, params=(SOURCE,))
labels_preview


,source,sentence_id,metaphor_category,metaphor_present,stance,confidence,reasoning,model_name,labeled_at,granularity


## 14. Idempotency check

After inserting the test batch, rerun this. The same `sentence_id` rows should not appear again as unlabeled.


In [26]:
remaining_rows = fetch_unlabeled_rows(conn, limit=20)

print("Next unlabeled rows:", len(remaining_rows))
pd.DataFrame(remaining_rows).head()


Next unlabeled rows: 20


,sentence_id,preceding_sentence,ai_sentence,subsequent_sentence,context_text,subreddit,created_utc,score
0,10074g2_s0000,None,AI Generated Content and Reddit - With the eme...,", what is Reddit position on using this conten...",AI Generated Content and Reddit - With the eme...,AskReddit,1672531224,1
1,10074g2_s0002,", what is Reddit position on using this conten...","For example, I used a book review generated by...",None,", what is Reddit position on using this conten...",AskReddit,1672531224,1
2,100753i_s0000,None,AI Generated Content and Reddit - With the eme...,", what is Reddit position on using this conten...",AI Generated Content and Reddit - With the eme...,AskReddit,1672531280,1
3,100753i_s0002,", what is Reddit position on using this conten...","For example, I used a book review generated by...",None,", what is Reddit position on using this conten...",AskReddit,1672531280,1
4,1007ad8_s0000,None,What are the potential consequences of artific...,None,What are the potential consequences of artific...,AskReddit,1672531735,1


## 15. Optional: parallel test batch

This simulates the production script more closely.


In [27]:
def process_batch_debug(
    rows: list[dict[str, Any]],
    metaphor_prompt_template: str,
    stance_prompt_template: str,
) -> list[dict[str, Any]]:
    labeled_rows = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(
                label_one_row,
                row,
                metaphor_prompt_template,
                stance_prompt_template,
            )
            for row in rows
        ]

        for future in as_completed(futures):
            labeled_rows.append(future.result())

    return labeled_rows


parallel_rows = fetch_unlabeled_rows(conn, limit=20)

parallel_results = process_batch_debug(
    rows=parallel_rows,
    metaphor_prompt_template=metaphor_prompt_template,
    stance_prompt_template=stance_prompt_template,
)

parallel_df = pd.DataFrame(parallel_results)

parallel_df[
    [
        "sentence_id",
        "metaphor_category",
        "granularity",
        "stance",
        "latency_seconds",
        "error",
    ]
].head()


,sentence_id,metaphor_category,granularity,stance,latency_seconds,error
0,10074g2_s0002,None,General-AI,Neutral/Unclear,1.925205,None
1,10074g2_s0000,None,General-AI,Neutral/Unclear,3.462368,None
2,100753i_s0000,None,General-AI,Neutral/Unclear,2.158697,None
3,100753i_s0002,None,General-AI,Neutral/Unclear,2.215677,None
4,1007ad8_s0000,Genie,General-AI,Positive,2.987411,None


## 16. Optional: insert parallel batch

Run only if the parallel outputs look valid.


In [28]:
# Uncomment when ready:
inserted_parallel = insert_labels(conn, parallel_results)
print("Inserted parallel batch:", inserted_parallel)


Inserted parallel batch: 20


## 17. Final readiness checklist

Before running `scripts/05_label_llm.py`, confirm:

- Ollama is running
- `llama3.2` is installed
- prompt files exist
- single-row test works
- 10-row test works
- DB insert works
- rerunning does not duplicate labels

Then run:

```bash
python scripts/05_label_llm.py
```


In [ ]:
# Close database connection when done.
# conn.close()
